# π0.5 LIBERO Replication — Colab (A100)

One unattended, **resumable** run: baseline eval → 30k-step LoRA fine-tune → eval every preserved checkpoint.

**If the session dies at any point: get a new A100 runtime and run all cells again.** Every stage
skips work that is already done (evals with a saved `results.json`, extracted LoRA files, finished
training), and training resumes from the newest checkpoint mirrored to Drive.

**Storage plan**

| Artifact | Where | Notes |
|---|---|---|
| Dataset (LeRobot tar, ~330 MB) | Drive → extracted to local SSD | `MyDrive/pi05_libero_replication/dataset/…tar` |
| Full checkpoints (~5 GB each) | local `/content`; newest mirrored to Drive | mirror enables cross-session training resume |
| LoRA adapters (~84 MB each) | Drive `lora/` | the real output |
| Eval metrics + videos | Drive `experiments/` + WandB | `results.json` on Drive doubles as the skip marker |
| Norm stats | git (committed in the repo) | — |
| All subprocess logs | Drive `logs/` | full child stdout/stderr, tee'd live into the cell |

**Why the install looks the way it does**
- openpi pins exact versions and pulls `lerobot`/`dlimp` from git → plain `pip install -e` backtracks
  forever against Colab's preinstalled stack. `uv sync` installs the lockfile into
  `third_party/openpi/.venv` (its own CPython 3.11). The Colab kernel only orchestrates; every ML step
  runs through that venv (`PY`) as a subprocess.
- **LIBERO is not pip-installed.** Its `setup.py` declares zero dependencies and its layout (no
  top-level `__init__.py`) breaks modern editable installs — an install "succeeds" but `import libero`
  fails. Upstream openpi puts the repo on `PYTHONPATH` instead; we do the same, and install its actual
  sim deps explicitly (`robosuite` with `--no-deps` to skip the keyboard-teleop stack whose `evdev`
  build fails headless). `mujoco==3.2.3` + `robosuite==1.4.1` is the combo openpi's own LIBERO example pins.
- Two import-time landmines are defused in §1b: LIBERO calls **`input()`** on first import if
  `~/.libero/config.yaml` is missing (instant `EOFError` in a subprocess), and torch ≥ 2.6's
  `weights_only=True` default breaks LIBERO's `torch.load` of init-state files
  (`TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1`).
- No `google.colab.auth`: the `gs://openpi-assets` bucket is public and openpi fetches it via
  anonymous `gsutil`.

**Time budget (A100 40 GB):** training ≈ 4–6 h; each eval (500 rollouts) ≈ 1.5–3 h; baseline + 6
checkpoints = 7 evals. That is likely **more than one session** — which is fine, rerun and it
continues. Lower `TRIALS_PER_TASK` (e.g. 20) for a faster full pass. If training OOMs, set
`BATCH_SIZE = 16`.

---
## §0 · Config

In [ ]:
# ── Tokens ────────────────────────────────────────────────────────────────────
WANDB_API_KEY = ""          # from wandb.ai/authorize (required)

# ── Sources ───────────────────────────────────────────────────────────────────
GITHUB_REPO = "https://github.com/LavetteSinsora/pi05-libero-replication"
BASE_CKPT   = "gs://openpi-assets/checkpoints/pi05_base"

# ── Fixed names (must match the TrainConfig) ────────────────────────────────────
CONFIG_NAME = "pi05_libero_object_lora"
EXP_NAME    = "masked_loss_summed_subsampling"
TOTAL_STEPS = 30_000              # config.num_train_steps
FINAL_STEP  = TOTAL_STEPS - 1     # train.py saves the last checkpoint at step 29999

# ── Paths ───────────────────────────────────────────────────────────────────────
REPO         = "/content/pi05-libero-replication"
OPENPI       = f"{REPO}/third_party/openpi"
LIBERO_DIR   = f"{REPO}/third_party/libero"
PY           = f"{OPENPI}/.venv/bin/python"            # venv python (built in §1c)
LEROBOT_HOME = "/content/lerobot"
DATASET_DIR  = f"{LEROBOT_HOME}/libero_object_summed_subsampling"
LOCAL_CKPT   = "/content/checkpoints/pi05_libero"      # full checkpoints (local SSD)
DRIVE_ROOT   = "/content/drive/MyDrive/pi05_libero_replication"
DRIVE_TAR    = f"{DRIVE_ROOT}/dataset/libero_object_summed_subsampling.tar"
DRIVE_LORA   = f"{DRIVE_ROOT}/lora/{EXP_NAME}"         # permanent LoRA adapters
DRIVE_CKPT   = f"{DRIVE_ROOT}/full_ckpt/{EXP_NAME}"    # rolling newest full ckpt (resume)
DRIVE_EXP    = f"{DRIVE_ROOT}/experiments"             # eval results + videos + skip markers
DRIVE_LOGS   = f"{DRIVE_ROOT}/logs"

# ── Knobs ───────────────────────────────────────────────────────────────────────
RUN_BASELINE       = True
TRIALS_PER_TASK    = 50     # 10 tasks × 50 = 500 rollouts per eval (~1.5-3 h each)
SYNC_CKPT_TO_DRIVE = True   # mirror newest full checkpoint (~5 GB) to Drive for resume
BATCH_SIZE         = None   # None → config default (32). Try 16 if training OOMs.

---
## §1 · Setup (idempotent — safe to rerun)

In [ ]:
# 1a · Mount Drive + clone the repo with both submodules
import os, pathlib, subprocess
from google.colab import drive

drive.mount("/content/drive")

if not pathlib.Path(REPO).exists():
    subprocess.run(["git", "clone", "--recurse-submodules", GITHUB_REPO, REPO], check=True)
subprocess.run(["git", "-C", REPO, "submodule", "update", "--init", "--recursive"], check=True)

assert WANDB_API_KEY, "set WANDB_API_KEY in §0"
assert pathlib.Path(DRIVE_TAR).exists(), (
    f"dataset tar not found at {DRIVE_TAR} — upload libero_object_summed_subsampling.tar "
    "(built with `tar -chf` so symlinked videos are dereferenced) to that Drive path first."
)
print("Repo + Drive ready.")

In [ ]:
# 1b · Environment (inherited by every subprocess) + LIBERO's config file
#
# - PYTHONPATH: LIBERO is not pip-installed (see header). With the repo on PYTHONPATH,
#   `libero` resolves as a namespace package — same trick upstream openpi uses.
# - ~/.libero/config.yaml: libero.libero.__init__ calls input() at import time if this
#   file is missing → EOFError in any non-interactive subprocess. Pre-write it.
# - TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD: torch>=2.6 defaults torch.load(weights_only=True),
#   which rejects LIBERO's .pruned_init files (pickled numpy) in get_task_init_states.
import yaml

os.environ["HF_LEROBOT_HOME"]                = LEROBOT_HOME
os.environ["MUJOCO_GL"]                      = "egl"    # GPU off-screen rendering
os.environ["PYOPENGL_PLATFORM"]              = "egl"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.9"
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
os.environ["PYTHONUNBUFFERED"]               = "1"      # live subprocess logs
os.environ["MPLBACKEND"]                     = "Agg"    # kernel exports the inline backend,
                                                        # which the venv's matplotlib lacks
os.environ["PYTHONPATH"]                     = LIBERO_DIR
os.environ["WANDB_API_KEY"]                  = WANDB_API_KEY

libero_pkg = f"{LIBERO_DIR}/libero/libero"
pathlib.Path(f"{LIBERO_DIR}/libero/datasets").mkdir(exist_ok=True)  # silence a path warning
cfg_dir = pathlib.Path.home() / ".libero"
cfg_dir.mkdir(exist_ok=True)
(cfg_dir / "config.yaml").write_text(yaml.dump({
    "benchmark_root": libero_pkg,
    "bddl_files":     f"{libero_pkg}/bddl_files",
    "init_states":    f"{libero_pkg}/init_files",
    "assets":         f"{libero_pkg}/assets",
    "datasets":       f"{LIBERO_DIR}/libero/datasets",
}))

for d in [LOCAL_CKPT, DRIVE_LORA, DRIVE_CKPT, DRIVE_EXP, DRIVE_LOGS]:
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)
print("Env set.")

In [ ]:
# 1c · System libs + Python env (uv sync against openpi's lockfile) + LIBERO sim deps
def run(cmd, cwd=None, env=None):
    """Stream the child's stdout+stderr into the cell — in Colab a plain
    subprocess.run() sends them to the server log, making failures unreadable."""
    print("$", " ".join(cmd), flush=True)
    p = subprocess.Popen(cmd, cwd=cwd, env=env, text=True, bufsize=1,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait():
        raise RuntimeError("command failed — traceback above: " + " ".join(cmd))

run(["apt-get", "update", "-qq"])
run(["apt-get", "install", "-y", "-qq", "ffmpeg", "libegl1", "libgl1", "libosmesa6"])
run(["pip", "install", "-q", "uv"])

# gsutil needs a *compiled* crcmod to download composite GCS objects (the pi05_base
# params are composite); with Colab's pure-python crcmod it refuses outright.
run(["pip", "install", "-q", "--no-cache-dir", "--force-reinstall", "--no-binary", ":all:", "crcmod"])
ext = subprocess.run(["python3", "-c", "import crcmod; print(crcmod.crcmod._usingExtension)"],
                     capture_output=True, text=True).stdout.strip()
if ext != "True":   # last resort: skip integrity checks rather than fail the download
    pathlib.Path.home().joinpath(".boto").write_text("[GSUtil]\ncheck_hashes = never\n")
    print("WARNING: compiled crcmod unavailable — disabled gsutil hash checks via ~/.boto")

# Exact locked deps (jax[cuda12], torch 2.7.1, lerobot@git, …) into third_party/openpi/.venv.
# uv fetches its own CPython 3.11 (openpi pins it); the Colab kernel stays on system Python.
run(["uv", "sync", "--no-dev"], cwd=OPENPI, env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"})

# LIBERO's sim deps — listed only in its requirements.txt, which pip/uv never read.
# numpy is re-pinned so a resolver bump of the locked version fails loudly here.
run(["uv", "pip", "install", "--python", PY,
     "numpy==1.26.4", "mujoco==3.2.3", "bddl==1.0.1", "future", "easydict",
     "gym==0.25.2", "cloudpickle", "matplotlib", "numba", "scipy", "termcolor", "h5py"])
run(["uv", "pip", "install", "--python", PY, "--no-deps", "robosuite==1.4.1"])

# Fail fast: the exact imports benchmark.py needs, inside the venv.
run([PY, "-c",
     "from libero.libero import benchmark, get_libero_path; "
     "from libero.libero.envs import OffScreenRenderEnv; "
     "import openpi.training.config, openpi_client; print('venv imports OK')"])

In [ ]:
# 1d · Dataset: extract the Drive tarball to local SSD.
# (HF snapshot_download of the 1,505-file repo repeatedly stalled at ~99% on Colab;
# one sequential tar read from Drive does not.) Completeness is checked by file
# counts — a partial dir would otherwise send lerobot to the HF Hub and fail confusingly.
import shutil, time

def dataset_complete(d):
    d = pathlib.Path(d)
    return ((d / "meta" / "info.json").exists()
            and len(list(d.glob("data/**/*.parquet"))) == 500
            and len(list(d.glob("videos/**/*.mp4"))) == 1000)

if dataset_complete(DATASET_DIR):
    print("Dataset already present and complete.")
else:
    shutil.rmtree(DATASET_DIR, ignore_errors=True)
    pathlib.Path(LEROBOT_HOME).mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    print("Extracting dataset tar from Drive (~330 MB)…")
    subprocess.run(["tar", "-xf", DRIVE_TAR, "-C", LEROBOT_HOME], check=True)
    assert dataset_complete(DATASET_DIR), "extraction incomplete — re-check the tar on Drive"
    print(f"Dataset extracted in {time.time() - t0:.0f}s.")

In [ ]:
# 1e · Rendering smoke test: build one LIBERO env off-screen and step it.
# Catches EGL/driver problems in ~1 min instead of hours into the run.
# (If this fails with an EGL error, retry with os.environ["MUJOCO_GL"] = "osmesa" —
# much slower, but a working fallback.)
smoke = """
import pathlib
from libero.libero import benchmark as B, get_libero_path
from libero.libero.envs import OffScreenRenderEnv
suite = B.get_benchmark_dict()["libero_object"]()
task = suite.get_task(0)
init = suite.get_task_init_states(0)
bddl = pathlib.Path(get_libero_path("bddl_files")) / task.problem_folder / task.bddl_file
env = OffScreenRenderEnv(bddl_file_name=str(bddl), camera_heights=256, camera_widths=256)
env.seed(7)
env.reset()
env.set_init_state(init[0])
obs, _, _, _ = env.step([0.0] * 6 + [-1.0])
assert obs["agentview_image"].shape == (256, 256, 3), obs["agentview_image"].shape
env.close()
print("EGL rendering OK — init states:", init.shape)
"""
run([PY, "-c", smoke])

In [ ]:
# 1f · Norm stats (committed in the repo) + anonymous read of the public GCS bucket.
norm_stats = pathlib.Path(
    f"{REPO}/assets/pi05_libero/{CONFIG_NAME}/libero_object_summed_subsampling/norm_stats.json"
)
assert norm_stats.exists(), (
    f"norm_stats.json missing at {norm_stats} — run compute_norm_stats.py locally "
    "and commit assets/ before training."
)
run(["gsutil", "ls", f"{BASE_CKPT}/"])   # public bucket; openpi downloads it via gsutil too
print("Setup complete.")

---
## §2 · Run the full experiment (unattended, resumable)

Stages, each **skipped automatically if already done**:
1. **Baseline eval** — pretrained π0.5 on LIBERO-OBJECT with *our* norm stats → WandB run
   `pi05_base_benchmark` + `results.json` on Drive.
2. **Train 30k steps** — one continuous run (checkpoints at 5k/10k/…/25k + final 29999 kept
   locally). A background thread mirrors the newest finished checkpoint to Drive every 5 min
   (rolling, keeps one ≈ 5 GB), so a dead session resumes instead of restarting.
3. **Extract LoRA adapters** from every preserved checkpoint → Drive. Runs *before* the long
   evals so the permanent artifacts are secured early.
4. **Eval every checkpoint** — 500 rollouts each → WandB run `step_<n>` (metrics + videos)
   + `results.json`/videos on Drive.

All subprocess output streams into this cell **and** into `Drive/…/logs/*.log`.

*Caveat:* intermediate checkpoints exist only on local disk. If the session dies after training,
a rerun restores the **final** checkpoint from Drive and skips finished evals; intermediate-step
evals need their local checkpoints (i.e. the same session) — their LoRA files are already safe.

In [ ]:
import shutil, subprocess, threading, time, pathlib

CKPT_RUN = pathlib.Path(LOCAL_CKPT) / CONFIG_NAME / EXP_NAME
DRIVE_CK = pathlib.Path(DRIVE_CKPT)

def sh(cmd, cwd=None, log_name="run", extra_env=None):
    """Run a command, streaming output live AND tee'ing it to a log file on Drive."""
    cmd = [str(c) for c in cmd]
    log = pathlib.Path(DRIVE_LOGS) / (log_name + ".log")
    print("\n$ " + " ".join(cmd) + "\n  [log: " + str(log) + "]", flush=True)
    with open(log, "a") as f:
        f.write("\n===== " + time.strftime("%F %T") + " $ " + " ".join(cmd) + "\n")
        p = subprocess.Popen(cmd, cwd=cwd, text=True, bufsize=1,
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             env={**os.environ, **(extra_env or {})})
        for line in p.stdout:
            print(line, end="", flush=True)
            f.write(line)
        rc = p.wait()
    if rc:
        raise RuntimeError(f"command failed (exit {rc}) — full log: {log}")

def local_steps():
    if not CKPT_RUN.exists():
        return []
    return sorted(int(d.name) for d in CKPT_RUN.iterdir() if d.is_dir() and d.name.isdigit())

def drive_steps():
    if not DRIVE_CK.exists():
        return []
    return sorted(int(d.name) for d in DRIVE_CK.iterdir() if d.is_dir() and d.name.isdigit())

def mirror_to_drive(step):
    """Copy one checkpoint to Drive (tmp dir + rename), then drop older mirrors."""
    tmp = DRIVE_CK / f"_tmp_{step}"
    shutil.rmtree(tmp, ignore_errors=True)
    shutil.copytree(CKPT_RUN / str(step), tmp)
    tmp.rename(DRIVE_CK / str(step))
    if (CKPT_RUN / "wandb_id.txt").exists():                 # needed by train.py --resume
        shutil.copy(CKPT_RUN / "wandb_id.txt", DRIVE_CK / "wandb_id.txt")
    for old in drive_steps():
        if old != step:
            shutil.rmtree(DRIVE_CK / str(old), ignore_errors=True)
    print(f"[sync] checkpoint {step} → Drive", flush=True)

def restore_from_drive():
    step = drive_steps()[-1]
    print(f"[restore] checkpoint {step}: Drive → local", flush=True)
    CKPT_RUN.mkdir(parents=True, exist_ok=True)
    shutil.copytree(DRIVE_CK / str(step), CKPT_RUN / str(step), dirs_exist_ok=True)
    if (DRIVE_CK / "wandb_id.txt").exists():
        shutil.copy(DRIVE_CK / "wandb_id.txt", CKPT_RUN / "wandb_id.txt")

def benchmark(checkpoint_dir, tag, train_step=None):
    exp_dir = pathlib.Path(DRIVE_EXP) / tag
    if (exp_dir / "results.json").exists():
        print(f"[skip] eval '{tag}' already done")
        return
    cmd = [PY, f"{REPO}/scripts/benchmark.py",
           "--config-name", CONFIG_NAME, "--checkpoint-dir", checkpoint_dir,
           "--exp-dir", exp_dir, "--num-trials-per-task", TRIALS_PER_TASK]
    if train_step is not None:
        cmd += ["--train-step", train_step]
    sh(cmd, cwd=OPENPI, log_name="eval_" + tag.replace("/", "_"))  # cwd → ../../assets resolves

# ── 1 · Baseline eval ──────────────────────────────────────────────────────────
if RUN_BASELINE:
    benchmark(BASE_CKPT, "pi05_base_benchmark")

# ── 2 · Train (skips if done; resumes across sessions via the Drive mirror) ────
if max(local_steps() + drive_steps(), default=-1) >= FINAL_STEP:
    if not local_steps():
        restore_from_drive()          # final ckpt back to local for the eval stage
    print("[skip] training already complete")
else:
    if not local_steps() and drive_steps():
        restore_from_drive()          # resume mid-training after a session loss
    if CKPT_RUN.exists() and not local_steps():
        shutil.rmtree(CKPT_RUN)       # leftover dir from a crash before the first save

    stop = threading.Event()
    def _sync_loop():
        synced = set(drive_steps())
        while not stop.wait(300):
            try:
                steps = local_steps()
                if steps and steps[-1] not in synced:
                    mirror_to_drive(steps[-1])
                    synced = {steps[-1]}
            except Exception as e:    # never kill training over a sync hiccup
                print(f"[sync] warning: {e}", flush=True)

    syncer = None
    if SYNC_CKPT_TO_DRIVE:
        syncer = threading.Thread(target=_sync_loop, daemon=True)
        syncer.start()

    cmd = [PY, "scripts/train.py", CONFIG_NAME,
           "--exp-name", EXP_NAME, "--checkpoint-base-dir", LOCAL_CKPT]
    if BATCH_SIZE:
        cmd += ["--batch-size", BATCH_SIZE]
    if local_steps():
        cmd.append("--resume")
    sh(cmd, cwd=OPENPI, log_name="train")

    if syncer:
        stop.set()
        syncer.join(timeout=5)
    if SYNC_CKPT_TO_DRIVE and local_steps() and local_steps()[-1] not in drive_steps():
        mirror_to_drive(local_steps()[-1])

# ── 3 · Secure LoRA adapters first (fast), then run the long evals ─────────────
steps = local_steps()
print(f"checkpoints to process: {steps}")

for s in steps:
    out = pathlib.Path(DRIVE_LORA) / f"step_{s}.npz"
    if out.exists():
        continue
    sh([PY, f"{REPO}/scripts/extract_lora.py",
        "--checkpoint-dir", CKPT_RUN / str(s), "--out", out],
       log_name="extract_lora", extra_env={"JAX_PLATFORMS": "cpu"})  # CPU-only restore

for s in steps:
    benchmark(CKPT_RUN / str(s), f"{EXP_NAME}/step_{s}", train_step=s)

print("\nDONE — metrics + videos in WandB, LoRA adapters + results on Drive.")

---
## §3 · (Optional) Results summary

All numbers are in WandB; this prints a table from the `results.json` files on Drive.

In [ ]:
import json, pathlib

rows = []
for p in sorted(pathlib.Path(DRIVE_EXP).rglob("results.json")):
    name = str(p.parent.relative_to(DRIVE_EXP))
    rows.append((name, json.loads(p.read_text())["aggregate_success_rate"]))

def order(row):
    name = row[0]
    return (0, 0) if "base" in name else (1, int(name.rsplit("_", 1)[-1]))

print(f"{'run':<52}{'success':>9}")
print("-" * 61)
for name, rate in sorted(rows, key=order):
    print(f"{name:<52}{rate:>8.1%}")